# Retinal Point-Spread Function

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from PIL import Image
from scipy.signal import fftconvolve
from scipy.fft import fft2, fftshift
from scipy.special import j1  # Bessel J1 for Airy PSF
import ipywidgets as widgets
from IPython.display import display

In [ ]:
SAMPLE_IMG_DIR = Path.cwd() / "imgs"

image_paths = sorted(
    p for p in SAMPLE_IMG_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
)

image_picker = widgets.Dropdown(
    options=[(p.name, p) for p in image_paths],
    description="Image: ",
    layout=widgets.Layout(width="500px"),
)
display(image_picker)

In [ ]:
GrayF = NDArray[np.float32]  # HxW in [0,1]

def load_grayscale(path: str | Path, max_size: int = 768) -> GrayF:
    """Load an image as grayscale float32 in [0, 1], thumbnailed to max_size."""
    img = Image.open(path).convert("L")
    img.thumbnail((max_size, max_size))
    return np.asarray(img, dtype=np.float32) / 255.0

## PSF Models

A **point-spread function** (PSF) describes how an optical system images a single point of light. Convolving an image with a PSF simulates the blur introduced by that system.

### Gaussian PSF
A simple isotropic Gaussian — not physically exact, but intuitive and widely used as an approximation:

$$
h(x, y) = \frac{1}{Z} \exp\!\left(-\frac{x^2 + y^2}{2\sigma^2}\right)
$$

### Airy PSF
The diffraction-limited PSF for a circular aperture (monochromatic). The intensity pattern is:

$$
h(r) = \left[\frac{2\, J_1(\rho)}{\rho}\right]^2, \quad \rho = \frac{3.8317 \cdot r}{r_{\text{Airy}}}
$$

where $J_1$ is the first-order Bessel function and $r_{\text{Airy}}$ is the first-zero radius in pixels.

In [ ]:
def gaussian_psf(size=129, sigma_px=2.0):
    """Simple blur kernel; not physically perfect but very intuitive."""
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    h = np.exp(-(xx**2 + yy**2) / (2 * sigma_px**2))
    h /= h.sum()
    return h


def airy_psf(size=257, airy_radius_px=3.0, eps=1e-12):
    """
    Diffraction-limited Airy PSF (monochromatic) in a convenient pixel-parameter form.

    airy_radius_px ~ first-zero radius in pixels (sets blur scale).
    """
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    r = np.sqrt(xx**2 + yy**2) + eps

    # Choose k so that first zero occurs at r = airy_radius_px.
    # First zero of J1 is at ~3.8317, and Airy amplitude uses 2*J1(x)/x.
    x = 3.8317 * (r / airy_radius_px)

    amp = 2 * j1(x) / (x + eps)
    h = amp**2
    h /= h.sum()
    return h


def convolve_image(image, psf):
    """FFT convolution; returns same-size result."""
    # fftconvolve returns full size unless we ask for 'same'
    return fftconvolve(image, psf, mode="same")


def show_results(I, h, title=""):
    Iret = convolve_image(I, h)
    diff = Iret - I

    # OTF / MTF
    H = fftshift(fft2(fftshift(h)))
    MTF = np.abs(H)
    MTF /= MTF.max() + 1e-12

    # PSF center slice
    c = h.shape[0] // 2
    psf_slice = h[c, :]

    fig = plt.figure(figsize=(12, 9))
    fig.suptitle(title, fontsize=14)

    # PSF (log)
    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(np.log10(h + 1e-12), cmap="gray")
    ax1.set_title("PSF (log10 intensity)")
    ax1.axis("off")

    ax2 = plt.subplot(2, 3, 2)
    ax2.plot(psf_slice)
    ax2.set_title("PSF center slice")
    ax2.set_xlabel("pixel")
    ax2.set_ylabel("intensity")

    ax3 = plt.subplot(2, 3, 3)
    ax3.imshow(MTF, cmap="gray")
    ax3.set_title("MTF (|OTF|, normalized)")
    ax3.axis("off")

    ax4 = plt.subplot(2, 3, 4)
    ax4.imshow(I, cmap="gray", vmin=0, vmax=1)
    ax4.set_title("Input image")
    ax4.axis("off")

    ax5 = plt.subplot(2, 3, 5)
    ax5.imshow(Iret, cmap="gray", vmin=0, vmax=1)
    ax5.set_title('"Retinal" image = input * PSF')
    ax5.axis("off")

    ax6 = plt.subplot(2, 3, 6)
    ax6.imshow(diff, cmap="gray")
    ax6.set_title("Difference (retinal - input)")
    ax6.axis("off")

    plt.tight_layout()
    plt.show()


# --- Use it ---
# Replace with your own image file:
I = load_grayscale("/Users/elij/Desktop/Imagery/di-img/fencing.jpg")

h = airy_psf(size=257, airy_radius_px=8.0)   # try 1.5, 2.5, 4.0
# h = gaussian_psf(size=129, sigma_px=2.0)   # simpler alternative

show_results(I, h, title="Convolution-based retinal image visualization")
